# Practice: TEME -> GCRF

A sandbox for `orbit_correction.frames`. It takes a real element set (TIMED, from the
committed CelesTrak-derived fixture), propagates it with SGP4 to get a TEME state, converts
it to GCRF, and then changes one thing at a time to see how far the GCRF position moves:

1. how far apart TEME and GCRF are at all;
2. the epoch the TEME vector belongs to;
3. the Earth orientation data (UT1-UTC and polar motion).

Nothing here writes to `src/` or `data/`. The EOP experiments swap astropy's table in memory
and restore the project's pinned table at the end.

Run it with the project environment (`uv run jupyter lab` after `uv add --dev jupyterlab`, or
select the `$HOME/.venvs/orbit-correction-screening` interpreter as the kernel in your editor).

In [1]:
import json
from pathlib import Path

import numpy as np
from astropy import units as u
from astropy.utils import iers
from sgp4 import omm
from sgp4.api import Satrec

from orbit_correction import frames as F

REPO = Path.cwd().parent if Path.cwd().name == "practice" else Path.cwd()
recs = json.loads((REPO / "tests/fixtures/celestrak_gp_sample.json").read_text())
timed = next(r for r in recs if r["NORAD_CAT_ID"] == 26998)
sat = Satrec()
omm.initialize(sat, timed)
epoch_tt = F.elset_epoch_to_tt_seconds(timed["EPOCH"])
print(timed["OBJECT_NAME"], timed["EPOCH"], "UTC  ->", epoch_tt, "TT s since J2000")
print("EOP in use:", F.eop_provenance()["kind"], F.eop_provenance()["astropy_iers_data"])

TIMED 2026-11-22T19:48:12.950496 UTC  -> 848648962.134496 TT s since J2000
EOP in use: astropy-iers-data bundled IERS-A + IERS-B 0.2026.11.23.0.35.53


## 1. How far apart are TEME and GCRF?

TEME is a frame *of date*: its z-axis is the true pole of the date and its x-axis the mean
equinox. GCRF's axes are fixed near the J2000 pole and equinox. About 26 years of precession
(roughly 50 arcsec per year) separates them, so the same numbers read as TEME or as GCRF
describe positions tens of kilometres apart.

In [2]:
err, r_teme, v_teme = sat.sgp4_tsince(0.0)
r_teme, v_teme = np.array(r_teme), np.array(v_teme)
r_gcrf, v_gcrf = F.teme_to_gcrf(r_teme, v_teme, epoch_tt)
print("TEME r [km]:", r_teme)
print("GCRF r [km]:", r_gcrf)
print(f"|r_GCRF - r_TEME| = {np.linalg.norm(r_gcrf - r_teme):.1f} km")
angle = np.degrees(np.arccos(r_teme @ r_gcrf / np.linalg.norm(r_teme) / np.linalg.norm(r_gcrf)))
print(f"angle between them: {angle * 3600:.0f} arcsec")

TEME r [km]: [ 6.69925438e+02 -6.92853673e+03  1.58021828e-03]
GCRF r [km]: [ 6.28243553e+02 -6.93244035e+03 -1.46213585e+00]
|r_GCRF - r_TEME| = 41.9 km
angle between them: 1241 arcsec


## 2. Change the epoch

Keep the TEME numbers fixed and change only the instant they are said to belong to. Because
TEME rotates (slowly) with the precession and nutation of the date, the GCRF answer moves.
Try other offsets, including whole decades.

In [3]:
for years in (-20, -5, -1, 0, 1):
    t = epoch_tt + years * 365.25 * 86400.0
    try:
        r_g, _ = F.teme_to_gcrf(r_teme, v_teme, t)
    except F.EOPCoverageError as e:
        print(f"{years:+3d} yr: {e}")
        continue
    print(f"{years:+3d} yr: GCRF position moves {np.linalg.norm(r_g - r_gcrf):8.3f} km")

-20 yr: GCRF position moves   31.164 km
 -5 yr: GCRF position moves    7.788 km
 -1 yr: GCRF position moves    1.561 km
 +0 yr: GCRF position moves    0.000 km
 +1 yr: time(s) MJD 61732.075-61732.075 outside the EOP table (MJD 41684.0-61728.0); see frames.eop_provenance()


A one-hour mistake in the epoch (for example mixing UTC and local time) is a different kind of
error: the frame barely moves in an hour, but the *satellite* does. Converting with the wrong
time scale (UTC treated as TT, 69 s apart) moves the frame by almost nothing; propagating to the
wrong time moves the satellite about 7.5 km per second of error.

In [4]:
r_g69, _ = F.teme_to_gcrf(r_teme, v_teme, epoch_tt - 69.184)
slip_m = np.linalg.norm(r_g69 - r_gcrf) * 1e3
print(f"frame-only effect of a 69.184 s time-scale slip: {slip_m:.3f} m")
print(f"satellite motion in 69.184 s: {np.linalg.norm(v_teme) * 69.184:.1f} km")

frame-only effect of a 69.184 s time-scale slip: 0.004 m
satellite motion in 69.184 s: 523.7 km


## 3. Change the Earth orientation data

SGP4's TEME is tied to Earth rotation through GMST, so astropy converts TEME -> GCRF *through*
the Earth-fixed frame (TEME -> ITRS -> CIRS -> GCRS), and both legs use UT1-UTC and polar motion.
Shift one EOP quantity and watch two positions: the Earth-fixed (ITRS) one and the GCRF one.

The table is a plain `IERS_A` copy (astropy's `IERS_Auto` re-reads its file, so edits to it
don't stick). The project's pinned table is restored at the end.

In [5]:
from astropy.coordinates import GCRS, ITRS, TEME, CartesianRepresentation


def positions(r_km, t_tt):
    t = F.tt_seconds_to_time(t_tt)
    c = TEME(CartesianRepresentation(r_km * u.km), obstime=t)
    itrs = c.transform_to(ITRS(obstime=t)).cartesian.xyz.to_value(u.km)
    gcrs = c.transform_to(GCRS(obstime=t)).cartesian.xyz.to_value(u.km)
    return itrs, gcrs


def with_shifted_eop(d_ut1_s=0.0, d_pm_arcsec=0.0):
    table = iers.IERS_A.open(iers.IERS_A_FILE).copy()
    table["UT1_UTC"] = table["UT1_UTC"] + d_ut1_s * u.s
    table["PM_x"] = table["PM_x"] + d_pm_arcsec * u.arcsec
    iers.earth_orientation_table.set(table)
    try:
        return positions(r_teme, epoch_tt)
    finally:
        F.configure_offline_eop()  # restore the project's pinned table


i0, g0 = with_shifted_eop()
for d_ut1, d_pm in ((0.001, 0), (0.01, 0), (0.1, 0), (0, 0.1), (0, 1.0)):
    i1, g1 = with_shifted_eop(d_ut1, d_pm)
    di_km, dg_m = np.linalg.norm(i1 - i0), np.linalg.norm(g1 - g0) * 1e3
    print(f"UT1-UTC {d_ut1 * 1e3:+6.0f} ms, x_p {d_pm:+4.1f} as:  "
          f"ITRS moves {di_km:8.4f} km,  GCRF moves {dg_m:.6f} m")
print("restored:", F.eop_provenance()["kind"])

UT1-UTC     +1 ms, x_p +0.0 as:  ITRS moves   0.0005 km,  GCRF moves 0.000000 m


UT1-UTC    +10 ms, x_p +0.0 as:  ITRS moves   0.0051 km,  GCRF moves 0.000000 m
UT1-UTC   +100 ms, x_p +0.0 as:  ITRS moves   0.0508 km,  GCRF moves 0.000005 m


UT1-UTC     +0 ms, x_p +0.1 as:  ITRS moves   0.0004 km,  GCRF moves 0.000000 m


UT1-UTC     +0 ms, x_p +1.0 as:  ITRS moves   0.0039 km,  GCRF moves 0.000000 m
restored: astropy-iers-data bundled IERS-A + IERS-B


The Earth-fixed position moves (about 0.5 m per millisecond of UT1; up to about 35 m per arcsecond of
polar motion, depending on where the satellite is), but the GCRF position does not: the EOP error enters TEME -> ITRS
and leaves again in ITRS -> GCRS. So for *this* conversion the choice of EOP file barely
matters. Where EOP does matter is anything Earth-fixed: the truth propagator's gravity field and
atmosphere, and any comparison with ground-station or ITRF data.

## Things to try

- Swap in another fixture object (IRAS 13777 at ~890 km, CZ-4C R/B 36123 on an eccentric
  orbit) and repeat section 1. Does the TEME-GCRF gap scale with radius?
- In section 3, convert an *ITRS* state to GCRF instead (as a ground-based observation would
  need). Does the UT1 sensitivity come back?
- Load the EOP file that the truth propagator will use with
  `F.use_eop_file(".../orekit-data/Earth-Orientation-Parameters/IAU-2000/finals2000A.all")`
  and compare against the bundled table at a recent epoch, where one file is predicted and the
  other measured. Call `F.configure_offline_eop()` afterwards.